# ProverbsLM Training — Google Colab

Trains a **58M parameter coding model** on code data.
- Runtime: T4 GPU (free tier)
- Time: ~4-6 hours for `small`, ~12-24h for `medium`
- Output: `.pt` checkpoint → convert to `.gguf` → load in Proverbs server

## Steps
1. Upload the `/Users/Stizzop/proverbs` folder as a zip, or clone from GitHub
2. Run all cells in order
3. Download `best.pt` when done
4. On your Mac: place it in `~/.proverbs/checkpoints/` and run `/train export`

In [ ]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else 'No GPU detected — switch runtime to GPU first')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install torch --index-url https://download.pytorch.org/whl/cu121 -q
!pip install datasets tqdm numpy fastapi uvicorn pydantic huggingface_hub -q
print('Dependencies installed.')

In [ ]:
# ── 3. Upload and extract Proverbs project ────────────────────────────────────
# Option A: Upload a zip file
from google.colab import files
print('Upload your proverbs project as a .zip file')
print('Zip it: cd /Users/Stizzop && zip -r proverbs.zip proverbs/ -x "*/node_modules/*" -x "*/dist/*" -x "*/venv*" -x "*/__pycache__/*"')
uploaded = files.upload()

import zipfile, os
for filename in uploaded:
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall('/content/')
        print(f'Extracted {filename}')

# Find the proverbs directory
import glob
dirs = glob.glob('/content/proverbs*')
PROVERBS_DIR = dirs[0] if dirs else '/content/proverbs'
print(f'Project at: {PROVERBS_DIR}')
os.chdir(PROVERBS_DIR)

In [ ]:
# ── 4. Download code pre-training data ───────────────────────────────────────
import sys
sys.path.insert(0, PROVERBS_DIR)

!python scripts/download_pretrain_data.py \
    --langs python javascript typescript \
    --max-per-lang 100000 \
    --output /root/.proverbs/pretrain_data

# Check what we got
import os
data_dir = '/root/.proverbs/pretrain_data'
for f in os.listdir(data_dir):
    if f.endswith('.jsonl'):
        size = os.path.getsize(os.path.join(data_dir, f))
        lines = sum(1 for _ in open(os.path.join(data_dir, f)))
        print(f'  {f}: {lines:,} examples ({size/1e6:.1f} MB)')

In [ ]:
# ── 5. Train the tokenizer on code data ──────────────────────────────────────
import os
tokenizer_path = '/root/.proverbs/tokenizer.json'

if not os.path.exists(tokenizer_path):
    !python -m tokenizer.train_tokenizer \
        --data-dir /root/.proverbs/pretrain_data \
        --vocab-size 32000 \
        --output {tokenizer_path}
else:
    print(f'Tokenizer already exists at {tokenizer_path}')

In [ ]:
# ── 6. Configure training ─────────────────────────────────────────────────────
# Choose your size based on available VRAM:
#   small  = 58M params, needs ~4GB VRAM (T4 free tier ✓)
#   medium = 254M params, needs ~8GB VRAM (T4 may OOM, use A100)
MODEL_SIZE = 'small'   # change to 'medium' on A100

# Training steps: more = better quality, longer time
#   small/T4: 50k steps = ~4-6h
MAX_STEPS  = 50000
BATCH_SIZE = 16        # reduce to 8 if OOM

print(f'Model: {MODEL_SIZE}, Steps: {MAX_STEPS:,}, Batch: {BATCH_SIZE}')

In [ ]:
# ── 7. Pre-train on code data ─────────────────────────────────────────────────
# This is the main training run. Watch the loss drop:
#   ~10.5 = random init
#   ~3.0  = learning code structure
#   ~2.0  = generating mostly valid syntax
#   ~1.5  = generating coherent, runnable code  ← target

!python -m training.train \
    --mode pretrain \
    --size {MODEL_SIZE} \
    --data /root/.proverbs/pretrain_data \
    --tokenizer /root/.proverbs/tokenizer.json \
    --output-dir /root/.proverbs/checkpoints \
    --batch-size {BATCH_SIZE} \
    --max-steps {MAX_STEPS} \
    --compile \
    --log-every 20 \
    --eval-every 500 \
    --save-every 2000

In [ ]:
# ── 8. Fine-tune on your coding sessions (optional) ───────────────────────────
# Upload your session files from ~/.proverbs/sessions/ first
from google.colab import files
print('Upload your ~/.proverbs/sessions/*.jsonl files for personal fine-tuning')
print('(Skip this cell if you want the base pre-trained model only)')

import os
os.makedirs('/root/.proverbs/sessions', exist_ok=True)
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = f'/root/.proverbs/sessions/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'Saved: {dest}')

session_count = len(list(os.listdir('/root/.proverbs/sessions')))
print(f'Session files: {session_count}')

In [ ]:
# ── 9. Fine-tune (runs only if sessions were uploaded) ────────────────────────
import os, glob
sessions = glob.glob('/root/.proverbs/sessions/*.jsonl')
pretrain_ckpt = '/root/.proverbs/checkpoints/pretrain/best.pt'

if sessions and os.path.exists(pretrain_ckpt):
    !python -m training.train \
        --mode finetune \
        --resume {pretrain_ckpt} \
        --sessions /root/.proverbs/sessions \
        --tokenizer /root/.proverbs/tokenizer.json \
        --output-dir /root/.proverbs/checkpoints \
        --batch-size 4 \
        --max-steps 3000 \
        --lr 1e-4
else:
    print('Skipping fine-tune — no sessions uploaded or pretrain checkpoint missing')

In [ ]:
# ── 10. Download the trained model ────────────────────────────────────────────
import os, glob
from google.colab import files

# Find the best checkpoint
candidates = [
    '/root/.proverbs/checkpoints/finetune/best.pt',
    '/root/.proverbs/checkpoints/pretrain/best.pt',
]
ckpt = next((p for p in candidates if os.path.exists(p)), None)

if ckpt:
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'Downloading: {ckpt}  ({size_mb:.0f} MB)')
    files.download(ckpt)
    
    # Also download tokenizer
    files.download('/root/.proverbs/tokenizer.json')
    
    print()
    print('On your Mac:')
    print('  cp best.pt ~/.proverbs/checkpoints/pretrain/best.pt')
    print('  cp tokenizer.json ~/.proverbs/tokenizer.json')
    print('  # Then in Proverbs CLI:')
    print('  /train export   # converts to .gguf')
    print('  # Restart the server — it will auto-load your trained model')
else:
    print('No checkpoint found — check training completed successfully')

## After downloading

1. Place `best.pt` in `~/.proverbs/checkpoints/pretrain/best.pt` on your Mac
2. In Proverbs: `/train export` — converts to GGUF
3. The server auto-discovers new `.gguf` files in `~/.proverbs/models/`
4. Restart the inference server: `launchctl kickstart -k gui/$(id -u)/com.proverbs.server`

Your model. Your weights. No external services.